# Best Model per Family on IEEE-CIS (After M2)
Note: this notebook is kept for after M2. It still trains one model on all training days (the M0 / M1 setup), so before running it the folds and the final refit should be changed to the M2 retraining setup

Logistic regression (LR), random forest (RF) and CatBoost learn in very different ways, so comparing a default LR with a tuned CatBoost would not be fair => we find the best version of each model family and compare the three best models on the validation rows

The notebook covers:
- **Best model of each family**: chosen using the training rows only. We remove features that stop ranking fraud later in time, try a small grid of settings (with and without class weights), test dropping correlated `V`, `C` and `D` columns, and remove feature groups the model does not use

- **Card history check**: we refit each best model without the card-history features. In IEEE-CIS, once a card has a chargeback, its later transactions are also labelled fraud, so features that recognise a card could raise the score without describing the transaction itself. The raw and reconstructed card IDs are never model inputs

- **SHAP**: SHAP values for the best CatBoost model

The features come from `04_feature_engineering.ipynb` (saved in `data/processed/`). We only use the training and validation rows here. The test rows are kept aside for the final evaluation, which is run once after all choices are fixed

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from catboost import CatBoostClassifier, Pool
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

SEED = 100
PROCESSED_DIR = Path("../data/processed")

## 1. Load the Features
We load the features and the feature lists saved by notebook 04 (run notebooks 03 and 04 first if these files are missing)
- Only the training and validation rows are kept, so the test rows never reach a model

- The model inputs are the columns in `ieee_feature_cols`. This list already leaves out the card IDs and other identifiers, which are only used to build features

- Missing values in categorical columns are replaced with the text `"missing"`, and these columns are stored as pandas categories (the models use this data type to find the categorical columns)

- `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` are stored as numbers **BUT** their values are codes (a larger number does not mean more of anything) => we turn them into text categories here, e.g. 315.0 becomes `"315"` and a missing value becomes `"missing"`, so every model gets them as categories

- Numeric columns are stored as float32 to save memory

- `ieee_v_reduced_drop` and `ieee_cd_corr_drop` are the correlated `V` and `C` / `D` columns from notebook 04, which we test dropping in Section 4.4

In [ ]:
features = pd.read_pickle(PROCESSED_DIR / "ieee_features.pkl")
with open(PROCESSED_DIR / "feature_lists.json") as f:
    feature_lists = json.load(f)

# these columns are stored as numbers, but the numbers are codes with no order
CODE_AS_CATEGORY = ["card1", "card2", "card3", "card5", "addr1", "addr2"]

feature_cols = feature_lists["ieee_feature_cols"]
cat_cols = [c for c in feature_cols if c in feature_lists["ieee_categorical_cols"] or c in CODE_AS_CATEGORY]
num_cols = [c for c in feature_cols if c not in cat_cols]
v_drop = feature_lists["ieee_v_reduced_drop"]
cd_drop = feature_lists["ieee_cd_corr_drop"]

# notebook 04 no longer builds the card1 + addr1 features, so rerun 04 if they are still in the list
old_cols = [c for c in feature_cols if "card1_addr1" in c]
if old_cols:
    raise RuntimeError(f"rerun notebook 04 first, these card1 + addr1 features are still in the feature list: {old_cols}")

# keep only the needed columns, and only the training and validation rows
keep = feature_cols + ["isFraud", "ieee_split", "TransactionDT"]
features = features.loc[features["ieee_split"].isin(["train", "validation"]), keep]
features[num_cols] = features[num_cols].astype("float32")
for col in cat_cols:
    if col in CODE_AS_CATEGORY:
        # numbers to text codes, e.g. 315.0 becomes "315" and a missing value becomes "missing"
        numbers = pd.to_numeric(features[col], errors="coerce")
        values = numbers.to_numpy(dtype="float64")
        rounded = np.round(values)
        whole = np.isfinite(values) & np.isclose(values, rounded)
        text = np.full(len(features), "missing", dtype=object)
        text[whole] = rounded[whole].astype("int64").astype(str)
        other = np.isfinite(values) & ~whole   # not a whole number => keep the value
        text[other] = numbers.to_numpy(dtype=object)[other]
        features[col] = pd.Series(text, index=features.index).astype("category")
    else:
        features[col] = features[col].astype("object").fillna("missing").astype(str).astype("category")

train = features[features["ieee_split"] == "train"]
val = features[features["ieee_split"] == "validation"]
del features

X_train, y_train = train[feature_cols], train["isFraud"]
X_val, y_val = val[feature_cols], val["isFraud"]
val_day = (val["TransactionDT"] // 86400).to_numpy()

print("training rows:", len(train), "| fraud rate:", round(y_train.mean() * 100, 2), "%")
print("validation rows:", len(val), "| fraud rate:", round(y_val.mean() * 100, 2), "% | days:", len(np.unique(val_day)))
print("features:", len(feature_cols), "| numeric:", len(num_cols), "| categorical (with the 6 code columns):", len(cat_cols), "| V columns in the drop list:", len(v_drop), "| C / D columns in the drop list:", len(cd_drop))

## 2. Evaluation Metrics
All models are scored on the validation rows with the same metrics (the full results table is in Section 5):
- **PR-AUC** (main metric): the area under the precision-recall curve, which does not depend on a threshold. A model that ranks transactions at random gets about the fraud rate, which is 0.039 on the validation rows, so this is the lowest score any model should get

- **ROC-AUC**

- **Precision, recall and F1** at a threshold of 0.5. We have not chosen a threshold yet, so 0.5 is just a default for now

- **Precision@100 per day**: for each validation day, we take the 100 transactions with the highest scores and calculate the share that are fraud, then average this over all validation days. 100 is about 3% of a day's transactions and stands for the number of cases a review team could check in one day. Note: the group has not decided yet whether k should count transactions or cards, so here it counts transactions

## 3. Models and Preprocessing
Each family gets the features in the form it can use. The column selectors pick the numeric and categorical columns by their data type, so the same pipeline works with any list of features. The preprocessing belongs to the family, so it is the same for M0, M1 and the best model of that family (M0 and M1 only differ in the class weights)

- **LR**: missing numeric values are filled with the training median, then we apply `arcsinh` and standardise each column. `arcsinh` works like a log for large values but is also defined at 0 and for negative values => this is important because many columns are very skewed and some `D` columns are negative. Categorical columns are one-hot encoded, and values with fewer than 100 training rows share one "infrequent" column (a weight learned from a few rows is mostly noise). The raw identity columns (e.g. `id_31`, `DeviceInfo`) are kept next to their cleaned versions (e.g. `id31_browser_family`, `device_brand`) => the cleaned ones give the general effect, and the raw ones add the effect of a specific common version or model. `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` are codes stored as categories (Section 1), so LR one-hot encodes them too. `card1` alone has about 12,000 codes => these six columns have their own one-hot encoder that only keeps codes with at least 500 training rows, and the rarer codes share one "infrequent" column

- **RF and CatBoost**: the same six code columns are given to the trees as categories, so a split can mean "this `card1` value" instead of "`card1` above 5000". The other numeric columns are used as they are, since both RF and CatBoost handle missing values themselves. RF ordinal-encodes the categorical columns: each category gets an integer learned from the training rows, missing is its own category, and a category that never appears in the training rows gets -1. CatBoost gets the categorical columns (`cat_cols`) through `cat_features` (`allow_writing_files=False` only stops CatBoost from writing log files and does not change the model)

`models` has one unfitted model per family. For each run we copy it with `clone` and only change the settings with `set_params`:
- **Class weights**: `class_weight="balanced"` for LR, `class_weight="balanced_subsample"` for RF (the weights are calculated inside each tree's bootstrap sample) and `auto_class_weights="Balanced"` for CatBoost

- **M0 and M1 settings**: `C` = 1 for LR, a minimum leaf size of 20 for RF, and depth 6 with 500 trees for CatBoost

- **LR iterations**: LR is fitted with lbfgs and up to 2000 iterations. On all training rows M0 converges after about 400 iterations **BUT** M1 needs about 1,000, because the class weights make the problem harder to solve => with a limit of 1000, M1 stopped just before converging

In [ ]:
numeric = make_column_selector(dtype_exclude="category")
categorical = make_column_selector(dtype_include="category")
code_pattern = r"^(?:card1|card2|card3|card5|addr1|addr2)$"
lr_codes = make_column_selector(pattern=code_pattern, dtype_include="category")   # the six code columns
lr_categorical = make_column_selector(pattern=r"^(?!(?:card1|card2|card3|card5|addr1|addr2)$)", dtype_include="category")   # all other categorical columns


# arcsinh without making another copy of the numeric columns (saves memory)
def arcsinh_in_place(X):
    return np.arcsinh(X, out=X)


models = {
    "LR": make_pipeline(
        ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(arcsinh_in_place), StandardScaler(copy=False)), numeric),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, dtype=np.float32), lr_categorical),   # rare values go into one column
            ("codes", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=500, dtype=np.float32), lr_codes),   # rare codes go into one column
        ]),
        LogisticRegression(max_iter=2000),   # M1 needs about 1,000 lbfgs iterations to converge on all training rows
    ),
    "RF": make_pipeline(
        ColumnTransformer([
            ("num", "passthrough", numeric),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical),
        ]),
        RandomForestClassifier(n_estimators=100, max_features="sqrt", n_jobs=-1, random_state=SEED),
    ),
    "CatBoost": CatBoostClassifier(learning_rate=0.1, eval_metric="PRAUC", random_seed=SEED, thread_count=-1,
                                   verbose=0, allow_writing_files=False),   # PR-AUC for early stopping (see fold_score)
}

# M0 and M1 settings, and the setting that turns on class weights for each family
base_settings = {
    "LR": {"logisticregression__C": 1.0},
    "RF": {"randomforestclassifier__min_samples_leaf": 20},
    "CatBoost": {"depth": 6, "iterations": 500},
}
class_weight_settings = {
    "LR": {"logisticregression__class_weight": "balanced"},
    "RF": {"randomforestclassifier__class_weight": "balanced_subsample"},
    "CatBoost": {"auto_class_weights": "Balanced"},
}


## 4. Best Model per Family
We compare the three families at their best settings. To choose these settings without touching the validation rows, we use expanding-window validation on the training rows (days 1 to 109):
- Each fold trains on all training days up to a cut-off day and scores the next 28 days

- The training window keeps all earlier days (instead of a fixed length), because the final model is also trained on all training days

- 28 days is close to the 32 days of the validation rows, and is full 4 weeks so each window has the same number of weekdays and weekends

Fitting every model on many folds would take too long, so we first train the M0 models with cut-offs every 7 days from day 46 to day 81 (6 cut-offs, the last window ends on day 109). Then we pick 3 cut-offs that are spread over the training period and whose mean PR-AUC is close to the mean of all 6. Nearby cut-offs share most of their scored days, so the extra cut-offs would mostly repeat the same evaluation

Every choice below uses the mean PR-AUC over the 3 folds (or the mean ROC-AUC for time consistency):
1. **Time consistency** (4.2): remove features whose relationship with fraud in a fold's training days does not hold in its scored days

2. **Tuning** (4.3): try a small grid of settings for each family, with and without class weights

3. **Correlated `V`, `C` and `D` columns** (4.4): test the two drop lists from notebook 04, and only drop a list if the model scores at least as well without it

4. **Group selection** (4.5): remove the feature groups that the tuned model does not use

5. **Refit** (4.6): train the chosen settings on all training rows

6. **Card history check** (4.7): refit the same model without the card-history features

The validation rows are not used for any of these choices. They are only scored after the choices are made, to report the results

In [ ]:
train_day = (train["TransactionDT"] // 86400).to_numpy()
WINDOW = 28
CUTOFFS = list(range(46, 82, 7))   # each fold scores the 28 days after its cut-off, and the last one ends on day 109

# each cut-off: train on all days up to the cut-off, score the next 28 days
fold_masks = {cutoff: (train_day <= cutoff, (train_day > cutoff) & (train_day <= cutoff + WINDOW)) for cutoff in CUTOFFS}


# 1. Rows in each fold

rows = []
for cutoff in CUTOFFS:
    fit_mask, score_mask = fold_masks[cutoff]
    rows.append({"cut-off day": cutoff, "trained on days": f"1-{cutoff}", "scored on days": f"{cutoff + 1}-{cutoff + WINDOW}",
                 "training rows": int(fit_mask.sum()), "scored rows": int(score_mask.sum()),
                 "fraud rate in scored rows": round(y_train[score_mask].mean(), 4)})
display(pd.DataFrame(rows).set_index("cut-off day"))


# 2. M0 PR-AUC at each cut-off

cutoff_check = {}
for family in ["LR", "RF", "CatBoost"]:
    for cutoff in CUTOFFS:
        start = time.time()
        fit_mask, score_mask = fold_masks[cutoff]
        model = clone(models[family]).set_params(**base_settings[family])
        if family == "CatBoost":
            model.fit(X_train[fit_mask], y_train[fit_mask], cat_features=cat_cols)
        else:
            model.fit(X_train[fit_mask], y_train[fit_mask])
        cutoff_check[(family, cutoff)] = average_precision_score(y_train[score_mask], model.predict_proba(X_train[score_mask])[:, 1])
        print(f"M0 {family}, cut-off day {cutoff}: PR-AUC {cutoff_check[(family, cutoff)]:.4f} ({time.time() - start:.0f}s)")
        del model

cutoff_check = pd.Series(cutoff_check).unstack()
cutoff_check.columns.name = "cut-off day"
cutoff_check["mean of all 6"] = cutoff_check[CUTOFFS].mean(axis=1)
display(cutoff_check.round(4))

**Takeaway**
- To be written

In [ ]:
FOLD_CUTOFFS = [46, 60, 81]
folds = [fold_masks[cutoff] for cutoff in FOLD_CUTOFFS]

print("mean M0 PR-AUC over all 6 cut-offs vs the 3 chosen cut-offs:")
display(pd.DataFrame({"all 6": cutoff_check[CUTOFFS].mean(axis=1),
                      f"cut-offs {FOLD_CUTOFFS}": cutoff_check[FOLD_CUTOFFS].mean(axis=1)}).round(4))


# train on each fold and score its scored days
# returns the mean PR-AUC, the PR-AUC of each fold, the settings and (if asked) the fold models
# CatBoost stops early on each fold's scored days, and its number of trees becomes the mean over the folds
def fold_score(family, settings, cols, keep_models=False):
    scores, trees, fold_models = [], [], []
    for fit_mask, score_mask in folds:
        X_tr, y_tr = X_train.loc[fit_mask, cols], y_train[fit_mask]
        X_sc, y_sc = X_train.loc[score_mask, cols], y_train[score_mask]
        model = clone(models[family]).set_params(**settings)
        if family == "CatBoost":
            # early stopping on PR-AUC without the class weights, same as average_precision_score
            # catboost only allows use_weights=false when the rows have weights => only with class weights
            if "auto_class_weights" in settings:
                model.set_params(eval_metric="PRAUC:use_weights=false")
            model.fit(X_tr, y_tr, cat_features=[c for c in cat_cols if c in cols], eval_set=(X_sc, y_sc), early_stopping_rounds=50)
            best_iter = model.get_best_iteration()
            if best_iter is None or best_iter < 0:   # no best iteration => use all trees
                best_iter = max(model.tree_count_ - 1, 0)
            trees.append(int(best_iter) + 1)
        else:
            model.fit(X_tr, y_tr)
        scores.append(average_precision_score(y_sc, model.predict_proba(X_sc)[:, 1]))
        if keep_models:
            fold_models.append(model)
    settings = dict(settings)
    if trees:
        settings["iterations"] = int(round(np.mean(trees)))
    return float(np.mean(scores)), scores, settings, fold_models


### 4.1 Feature Groups
Each feature is put in one group based on what it describes:
- The `V` columns come in blocks that are missing for the same transactions, so we group them by their missing rate on the training rows

- The raw codes `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` form their own group, so the group selection (Section 4.5) shows whether each model uses the raw codes on top of their frequency columns and group statistics. LR gets them as one-hot categories (Section 3)

The card-history groups are needed for the card history check (Section 4.7). The best model is still allowed to use them, and Section 4.7 then measures how much its score depends on them

**Card history**: features built only from earlier transactions of the same `card_id`, never from the labels. Once a card has a chargeback, its later transactions are also labelled fraud => history features can help because they recognise a card whose earlier behaviour was unusual, and not only because they describe the current transaction
- `card window`: the number and total amount of the card's transactions in the past 1 hour, 24 hours, 7 days and 30 days, and the current amount compared with the card's earlier amounts
- `card past averages`: the mean and standard deviation of earlier values on the same `card_id`, and whether a value (e.g. the email domain) is new for this card
- `card history`: the number of earlier transactions on this `card_id`, the time since the previous one, and whether `D3` matches that gap. `card_id` was partly built from `D3` in notebook 03, so the match is high for many rows by construction

**Kept in every model**, because many different cards share these keys:
- `coarse group statistics`: the mean and standard deviation of `TransactionAmt`, `D1`, `D10` and `D15` within each `card1` group, calculated on the training rows
- `coarse key history`: the number of earlier transactions and the time since the previous one, by `card1`, `addr1`, an email domain or `DeviceInfo`
- `frequency`: frequencies of single columns such as `card1`, `addr1`, the email domains, `ProductCD` and `DeviceInfo`

In [ ]:
v_missing = X_train[[c for c in feature_cols if c.startswith("V") and c[1:].isdigit()]].isna().mean().round(2)

window_cols = {"is_first_txn", "amt_to_prior_mean", "amt_prior_zscore", "is_new_category"}
missing_cols = {"missing_count", "missing_fraction", "card_missing_count", "address_missing_count", "D_nonmissing_count", "D_zero_count"}
device_cols = {"DeviceType", "DeviceInfo", "device_brand", "id30_os_family", "id31_browser_family", "os_browser_pair", "has_identity",
               "screen_width", "screen_height", "screen_pixels"}
time_cols = {"hour", "minute", "seconds_in_day", "is_night", "hour_sin", "hour_cos", "day_sin", "day_cos"}

CARD_HISTORY_GROUPS = {"card window", "card past averages", "card history"}

group_of = {}
for col in feature_cols:
    if col in v_missing.index:
        group_of[col] = f"V (missing {v_missing[col]:.0%})"
    elif col in CODE_AS_CATEGORY:
        group_of[col] = "card and address codes"
    elif col.endswith("_by_card1"):
        group_of[col] = "coarse group statistics"
    elif col.endswith(("_card_past_mean", "_card_past_std", "_card_past_distinct", "_new_for_card")):
        group_of[col] = "card past averages"
    elif col.startswith(("txn_count_", "amt_sum_", "amt_mean_")) or col in window_cols:
        group_of[col] = "card window"
    elif col in {"card_id_previous_count", "card_id_seconds_since_previous"} or col.startswith(("D3_gap", "D3_matches")):
        group_of[col] = "card history"
    elif col.endswith(("_previous_count", "_seconds_since_previous")):
        group_of[col] = "coarse key history"
    elif col.endswith("_freq"):
        group_of[col] = "frequency"
    elif col in missing_cols or col.endswith(("missing_count", "missing_fraction", "_missing")):
        group_of[col] = "missingness"
    elif col.startswith("C") and col[1:].isdigit():
        group_of[col] = "C"
    elif (col.startswith("D") and col[1:].isdigit()) or col == "D3_log1p":
        group_of[col] = "D"
    elif col.startswith("M") and col[1:].isdigit():
        group_of[col] = "M"
    elif col.startswith("id_"):
        group_of[col] = "identity id_"
    elif col in device_cols or col.startswith("deviceinfo_"):
        group_of[col] = "device"
    elif "emaildomain" in col or col == "email_domain_match":
        group_of[col] = "email"
    elif col.startswith("card"):
        group_of[col] = "card"
    elif col.startswith(("addr", "dist", "address")):
        group_of[col] = "address"
    elif col.startswith(("TransactionAmt", "amount")):
        group_of[col] = "amount"
    elif col in time_cols:
        group_of[col] = "time"
    elif col == "ProductCD":
        group_of[col] = "product"
    else:
        group_of[col] = "other"

# role of each group (all other groups are "transaction-level")
group_role = {group: "card history" for group in CARD_HISTORY_GROUPS}
group_role["coarse group statistics"] = "coarse entity aggregate"
group_role["coarse key history"] = "coarse entity aggregate"

roles = pd.Series(group_of)
summary = pd.DataFrame({"group": roles, "role": [group_role.get(g, "transaction-level") for g in roles]})
display(summary.groupby(["role", "group"]).size().rename("features").to_frame())

# make sure no card-history or card ID column ends up in "other"
other_cols = [c for c, g in group_of.items() if g == "other"]
print("columns in 'other':", other_cols)
missed = [c for c in other_cols if "card_id" in c or c.startswith("D3_") or "card_past" in c or c.endswith("_by_card1")]
if missed:
    raise RuntimeError(f"card-history or card-id columns were not classified: {missed}")

### 4.2 Time Consistency
A feature can be related to fraud early in the training period and then lose that relationship (or reverse it) later on, which hurts the model on later data. For each feature on its own and for each fold, we train a small model (`HistGradientBoostingClassifier` with 50 trees) on the training days and calculate its ROC-AUC on the scored days, then take the mean over the 3 folds:
- **Mean ROC-AUC above 0.5**: the feature still ranks fraud better than random in the later periods => keep it
- **Mean ROC-AUC of 0.5 or lower**: the feature no longer helps, or works in the opposite direction => drop it

Note: `HistGradientBoostingClassifier` only accepts up to 255 categories per column, so categorical columns with more than 250 categories are passed as integer codes. The six code columns (`card1`, `card2`, `card3`, `card5`, `addr1`, `addr2`) are passed as their numbers in this check

In [ ]:
start = time.time()
consistency = {}

for col in feature_cols:
    x = X_train[[col]]
    if col in CODE_AS_CATEGORY:
        # the code columns go in as their numbers here
        x = pd.to_numeric(x[col].astype(str), errors="coerce").astype("float32").to_frame()
    elif col in cat_cols and len(X_train[col].cat.categories) > 250:
        # too many categories for HistGradientBoostingClassifier => use the integer codes
        x = x[col].cat.codes.to_frame()
    aucs = []
    for fit_mask, score_mask in folds:
        model = HistGradientBoostingClassifier(max_iter=50, categorical_features="from_dtype", early_stopping=False, random_state=SEED)
        model.fit(x[fit_mask], y_train[fit_mask])
        aucs.append(roc_auc_score(y_train[score_mask], model.predict_proba(x[score_mask])[:, 1]))
    consistency[col] = np.mean(aucs)

consistency = pd.Series(consistency, name="mean ROC-AUC over the folds").sort_values()
consistent_cols = [c for c in feature_cols if consistency[c] > 0.5]

print(f"kept {len(consistent_cols)} of {len(feature_cols)} features ({time.time() - start:.0f}s)")
print("dropped features by group:")
display(pd.Series([group_of[c] for c in feature_cols if c not in consistent_cols]).value_counts().rename("dropped").to_frame())
print("lowest ROC-AUC:")
display(consistency.head(10).round(4).to_frame())


### 4.3 Tuning
Using only the time-consistent features, we train each setting on the 3 folds and take its mean PR-AUC:
- **LR**: regularisation strength `C` of 0.1 or 1
- **RF**: minimum leaf size of 20 or 50
- **CatBoost**: tree depth of 6 or 8. The number of trees is set by early stopping on each fold's scored days (at most 2,000 trees, stopping after 50 trees in a row with no improvement), and the final model uses the mean number of trees over the 3 folds

Each setting is tried with and without class weights, and for each family we keep the setting with the highest mean PR-AUC

Note: for CatBoost, the scored days are used both for early stopping and for comparing the settings, so its fold scores are slightly optimistic (the validation rows are not affected)

In [ ]:
grids = {
    "LR": [{"logisticregression__C": C} for C in [0.1, 1.0]],
    "RF": [{"randomforestclassifier__min_samples_leaf": leaf} for leaf in [20, 50]],
    "CatBoost": [{"depth": depth, "iterations": 2000} for depth in [6, 8]],
}

tuning = []
best_settings, best_score, best_models = {}, {}, {}

for family, grid in grids.items():
    best_score[family] = -1
    for settings in grid:
        for weighted in [False, True]:
            start = time.time()
            settings_run = settings | (class_weight_settings[family] if weighted else {})
            score, scores, settings_run, fold_models = fold_score(family, settings_run, consistent_cols, keep_models=True)
            tuning.append({"family": family, "settings": settings_run, "mean PR-AUC": score, "PR-AUC per fold": scores})
            per_fold = ", ".join(f"{s:.4f}" for s in scores)
            print(f"{family} {settings_run}: PR-AUC {score:.4f} (folds: {per_fold}) ({time.time() - start:.0f}s)")

            # keep the setting with the highest mean PR-AUC
            if score > best_score[family]:
                best_settings[family], best_score[family], best_models[family] = settings_run, score, fold_models
            else:
                del fold_models

print("\nbest settings (chosen on the mean PR-AUC over the folds):")
for family in grids:
    print(f"  {family}: {best_settings[family]} (PR-AUC {best_score[family]:.4f})")

### 4.4 Correlated V, C and D Columns
Notebook 04 saves two drop lists:
- `V` columns with an absolute Spearman correlation above 0.75 with another `V` column that is missing in exactly the same training rows (221 columns)
- `C` and `D` columns with an absolute Spearman correlation above 0.9 with another `C` or `D` column (6 columns)

For each family we refit the tuned model on the 3 folds without the `V` list, and then also without the `C` / `D` list. A list is only dropped if the mean PR-AUC is at least as high as before, so a smaller set never costs accuracy on the folds. All three families get the same test => correlated columns mainly hurt LR, but we let the folds decide whether they help or hurt RF and CatBoost

In [ ]:
chosen_cols = {}
reduction_used = {}

for family in ["LR", "RF", "CatBoost"]:
    chosen_cols[family] = consistent_cols
    reduction_used[family] = {}
    for label, drop_list in [("V", v_drop), ("C / D", cd_drop)]:
        start = time.time()
        reduced_cols = [c for c in chosen_cols[family] if c not in drop_list]
        settings_run = best_settings[family] | ({"iterations": 2000} if family == "CatBoost" else {})
        score, scores, settings_run, fold_models = fold_score(family, settings_run, reduced_cols, keep_models=True)
        # only drop the list if the mean PR-AUC does not go down
        use_reduced = score >= best_score[family]
        reduction_used[family][label] = use_reduced
        per_fold = ", ".join(f"{s:.4f}" for s in scores)
        print(f"{family}: current {best_score[family]:.4f} | without the {label} drop list {score:.4f} (folds: {per_fold}, "
              f"{len(reduced_cols)} features) => {'drop them' if use_reduced else 'keep them'} ({time.time() - start:.0f}s)")
        if use_reduced:
            chosen_cols[family] = reduced_cols
            best_settings[family], best_score[family], best_models[family] = settings_run, score, fold_models
        else:
            del fold_models

display(pd.DataFrame(reduction_used).T.rename(columns=lambda c: f"drop the {c} list"))

### 4.5 Group Selection
For each tuned model and each fold, we shuffle one feature group at a time on the fold's scored days and measure how much the fold model's PR-AUC drops (permutation importance, repeated 5 times), then take the mean drop over the 3 folds and 5 repeats. All columns in a group are shuffled with the same row order, so they stay consistent with each other. If shuffling a group does not lower PR-AUC on average, the model does not use that group => drop it

**BUT** each group is shuffled on its own, so two groups that carry the same information can both look unused (the model falls back on the other one), and dropping them together could still hurt. So we retrain on the 3 folds without the dropped groups and only keep the smaller feature set if the mean PR-AUC does not go down, same rule as Section 4.4

We shuffle groups instead of single features because:
1. There are several hundred features, so shuffling them one at a time would take too long

2. Many features carry the same information (e.g. the `V` columns), so shuffling only one of them would show almost no drop even when that information matters

In [ ]:
rng = np.random.default_rng(SEED)
importance, selected_cols = {}, {}

for family in ["LR", "RF", "CatBoost"]:
    start = time.time()
    cols = chosen_cols[family]
    scored = [(X_train.loc[score_mask, cols], y_train[score_mask]) for _, score_mask in folds]
    bases = [average_precision_score(y_sc, model.predict_proba(X_sc)[:, 1])
             for model, (X_sc, y_sc) in zip(best_models[family], scored)]

    rows = []
    for group in sorted(set(group_of[c] for c in cols)):
        group_cols = [c for c in cols if group_of[c] == group]
        drops = []
        for model, base, (X_sc, y_sc) in zip(best_models[family], bases, scored):
            for repeat in range(5):
                perm = rng.permutation(len(X_sc))
                # shuffle all columns of the group with the same row order
                X_perm = X_sc.copy()
                for col in group_cols:
                    X_perm[col] = X_sc[col].iloc[perm].set_axis(X_sc.index)
                drops.append(base - average_precision_score(y_sc, model.predict_proba(X_perm)[:, 1]))
        rows.append({"group": group, "role": group_role.get(group, "transaction-level"), "features": len(group_cols),
                     "PR-AUC drop": np.mean(drops), "drop std": np.std(drops)})

    importance[family] = pd.DataFrame(rows).set_index("group").sort_values("PR-AUC drop", ascending=False)
    # keep the groups whose shuffling lowers PR-AUC on average
    keep = importance[family].index[importance[family]["PR-AUC drop"] > 0]
    reduced_cols = [c for c in cols if group_of[c] in keep]
    dropped_groups = [g for g in importance[family].index if g not in set(keep)]
    print(f"{family}: keep {len(keep)} of {len(importance[family])} groups, {len(reduced_cols)} features ({time.time() - start:.0f}s)")
    print("  dropped:", dropped_groups)
    del scored, best_models[family]   # free the fold models before retraining

    # retrain on the folds without the dropped groups, only keep them out if the mean PR-AUC does not go down
    selected_cols[family] = cols
    if dropped_groups:
        start = time.time()
        settings_run = best_settings[family] | ({"iterations": 2000} if family == "CatBoost" else {})
        score, scores, settings_run, _ = fold_score(family, settings_run, reduced_cols)
        use_reduced = score >= best_score[family]
        per_fold = ", ".join(f"{s:.4f}" for s in scores)
        print(f"  current {best_score[family]:.4f} | without the dropped groups {score:.4f} (folds: {per_fold}) "
              f"=> {'drop them' if use_reduced else 'keep them'} ({time.time() - start:.0f}s)")
        if use_reduced:
            selected_cols[family] = reduced_cols
            best_settings[family], best_score[family] = settings_run, score

del best_models
display(pd.concat({family: imp["PR-AUC drop"] for family, imp in importance.items()}, axis=1).round(4))

### 4.6 Final Models
We refit each tuned model on all training rows twice and score the validation rows:
1. With all candidate features

2. With the selected features (the time-consistent features, without the `V` and `C` / `D` lists if they were dropped, and without the unused groups if dropping them passed the fold check). This is the best version of each family

The best CatBoost model is kept for SHAP (Section 6), and Section 4.7 refits these models again without the card-history groups

In [ ]:
val_scores = {}   # validation scores of every model, all evaluated together in Section 5

for family in ["LR", "RF", "CatBoost"]:
    for label, cols in [("tuned, all features", feature_cols), ("tuned + selected", selected_cols[family])]:
        start = time.time()
        model = clone(models[family]).set_params(**best_settings[family])
        if family == "CatBoost":
            model.fit(X_train[cols], y_train, cat_features=[c for c in cat_cols if c in cols])
        else:
            model.fit(X_train[cols], y_train)
        val_scores[f"{family} {label}"] = model.predict_proba(X_val[cols])[:, 1]
        print(f"{family} {label}: PR-AUC {average_precision_score(y_val, val_scores[f'{family} {label}']):.4f} "
              f"({len(cols)} features, {time.time() - start:.0f}s)")

        # keep the best CatBoost model for SHAP (Section 6)
        if family == "CatBoost" and label == "tuned + selected":
            shap_model, shap_cols = model, cols
        else:
            del model

### 4.7 Card History Check
We report each best model with and without the card-history features. These features do not use the labels **BUT** they can still recognise a card whose earlier behaviour was unusual, and that card's later transactions are labelled fraud after a chargeback => this check shows how much of each score depends on that

For each family, we use its best settings and refit on all training rows with the selected features minus the card-history groups, then compare with the selected features (already scored in Section 4.6)

The coarse `card1` statistics and the history by email domain, `DeviceInfo` and `addr1` stay in, because many cards share those keys. The settings are not changed based on these validation scores


In [ ]:
# feature sets for the card history check
history_checks = [
    ("no card history", CARD_HISTORY_GROUPS),
]

for family in ["LR", "RF", "CatBoost"]:
    full = selected_cols[family]
    print(f"\n{family}: selected features {len(full)}")
    for label, groups in history_checks:
        cols = [c for c in full if group_of[c] not in groups]
        removed = [c for c in full if c not in cols]
        name = f"{family} tuned + selected, {label}"
        # nothing to remove => same model as the selected features
        if cols == full:
            val_scores[name] = val_scores[f"{family} tuned + selected"]
            print(f"  {label}: no columns of this kind in the selected set, same score")
            continue
        start = time.time()
        model = clone(models[family]).set_params(**best_settings[family])
        if family == "CatBoost":
            model.fit(X_train[cols], y_train, cat_features=[c for c in cat_cols if c in cols])
        else:
            model.fit(X_train[cols], y_train)
        val_scores[name] = model.predict_proba(X_val[cols])[:, 1]
        del model
        print(f"  {label}: drop {len(removed)} features, PR-AUC "
              f"{average_precision_score(y_val, val_scores[name]):.4f} ({time.time() - start:.0f}s)")

## 5. Validation Results
PR-AUC is the main metric. Precision, recall and F1 use the 0.5 threshold, so they change when class weights shift the scores, even if the ranking of transactions stays the same. Precision@100 per day is the share of fraud among each day's 100 highest-scored transactions, averaged over the validation days

To compare families, use the "tuned + selected" row of each family (its best model). The row below it uses the same settings without the card-history features

In [ ]:
k = 100
rows = []
for name, scores in val_scores.items():
    predictions = (scores >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, predictions, labels=[0, 1]).ravel()

    # precision@k per day: share of fraud among each day's k highest scores, averaged over the days
    daily = pd.DataFrame({"day": val_day, "y": y_val.to_numpy(), "score": scores})
    top_k = daily.sort_values("score", ascending=False, kind="stable").groupby("day").head(k)

    rows.append({"model": name,
                 "PR-AUC": average_precision_score(y_val, scores),
                 "ROC-AUC": roc_auc_score(y_val, scores),
                 "precision": precision_score(y_val, predictions, zero_division=0),
                 "recall": recall_score(y_val, predictions, zero_division=0),
                 "F1": f1_score(y_val, predictions, zero_division=0),
                 f"precision@{k} per day": top_k.groupby("day")["y"].mean().mean(),
                 "TP": tp, "FP": fp, "FN": fn, "TN": tn})

results_table = pd.DataFrame(rows).set_index("model")
display(results_table.round(4))

metric_cols = ["PR-AUC", "ROC-AUC", "precision", "recall", "F1", f"precision@{k} per day"]
best_rows = []
for family in ["LR", "RF", "CatBoost"]:
    best_rows += [
        f"{family} tuned + selected",
        f"{family} tuned + selected, no card history",
    ]
print("best of each family, then the same model without the card-history features:")
display(results_table.loc[best_rows, metric_cols].round(4))

print("family ranking by validation PR-AUC:")
for title, suffix in [
    ("best (selected features)", "tuned + selected"),
    ("without card history", "tuned + selected, no card history"),
]:
    ranking = sorted(
        ["LR", "RF", "CatBoost"],
        key=lambda family: results_table.loc[f"{family} {suffix}", "PR-AUC"],
        reverse=True,
    )
    bits = ", ".join(f"{family} {results_table.loc[f'{family} {suffix}', 'PR-AUC']:.4f}" for family in ranking)
    print(f"  {title}: {bits}")


In [ ]:
# precision-recall curves on the validation rows
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

curve_names = [
    "{family} tuned, all features",
    "{family} tuned + selected",
    "{family} tuned + selected, no card history",
]
for ax, family in zip(axes, ["LR", "RF", "CatBoost"]):
    for pattern in curve_names:
        name = pattern.format(family=family)
        precision, recall, thresholds = precision_recall_curve(y_val, val_scores[name])
        ax.plot(recall, precision, label=f"{name} ({results_table.loc[name, 'PR-AUC']:.3f})")

    # a model that ranks at random gets the fraud rate
    ax.axhline(y_val.mean(), color="grey", linestyle="--", label=f"random ({y_val.mean():.3f})")
    ax.set_title(f"{family}: precision-recall on validation")
    ax.set_xlabel("Recall")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Precision")
plt.tight_layout()
plt.show()

**Takeaway: Best Model per Family and Card History Check**
- To be written after the rerun

## 6. SHAP Values for the Best CatBoost Model
SHAP values split each prediction into a contribution from each feature, and a positive value pushes the score towards fraud. We calculate them with CatBoost's built-in SHAP for 5,000 random validation transactions and show:
- The features with the largest mean absolute SHAP value (the features the model relies on most)
- The total mean absolute SHAP value per feature group, to compare with the group selection in Section 4.5
- The direction of each numeric feature: the Spearman correlation between its value and its SHAP value (positive => higher values push the score towards fraud)

Note: SHAP only describes what this fitted model uses. It does not show whether a feature generalises to later data, so we do not use it to select features. If card-history features rank high here, the model partly relies on recognising the card (Section 4.7 measures how much the score depends on that)

In [ ]:
shap_rows = X_val[shap_cols].sample(5000, random_state=SEED)
shap_cats = [c for c in cat_cols if c in shap_cols]
shap_values = shap_model.get_feature_importance(Pool(shap_rows, cat_features=shap_cats), type="ShapValues")[:, :-1]
shap_values = pd.DataFrame(shap_values, index=shap_rows.index, columns=shap_cols)

mean_abs = shap_values.abs().mean().sort_values(ascending=False)
group_abs = mean_abs.groupby(pd.Series(group_of)).sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
mean_abs.head(20).iloc[::-1].plot.barh(ax=axes[0])
axes[0].set_title("Top 20 features by mean |SHAP|")
axes[0].set_xlabel("mean |SHAP| (log-odds)")
group_abs.iloc[::-1].plot.barh(ax=axes[1])
axes[1].set_title("Total mean |SHAP| per feature group")
axes[1].set_xlabel("sum of mean |SHAP| (log-odds)")
plt.tight_layout()
plt.show()

# top 20 features by mean |SHAP|, with their group and direction (categorical features have no direction)
top_shap = mean_abs.head(20).rename("mean |SHAP|").to_frame()
top_shap.insert(0, "group", [group_of[c] for c in top_shap.index])
top_shap.insert(1, "role", [group_role.get(group_of[c], "transaction-level") for c in top_shap.index])
top_shap["value vs SHAP (Spearman)"] = [
    pd.Series(shap_rows[c].to_numpy(), dtype="float64").corr(pd.Series(shap_values[c].to_numpy()), method="spearman")
    if c not in shap_cats else np.nan
    for c in top_shap.index
]
display(top_shap.round(3))


**Takeaway: SHAP**
- To be written